# 🚀 المربّي (Tutor Guardian) — Google Colab Ultra Pipeline (H100 / A100 80GB)
---
هذا النوتبوك مصمم خصيصاً لاستغلال اشتراك **Google AI Ultra** (كروت **H100 80GB** أو **A100 80GB** مع ميزة التشغيل في الخلفية) لتنفيذ خط الإنتاج الكامل أوتوماتيكياً بدون رفع يدوي:

1. **المرحلة 0 (السحب التلقائي من HuggingFace Hub):** سحب الحزمة الكاملة `tg_colab_ultra_bundle.zip` من المستودع الخاص `khaledhamdy/tg-ultra-v6-private` تلقائياً عبر `HF_TOKEN`.
2. **المرحلة 1 (GPU Retrieval & Rerank Probe):** فحص كل أسئلة الآباء والأمهات الحقيقية (`2,471` رسالة) على الـ GPU في ثوانٍ باستخدام `multilingual-e5-small` و `mmarco-mMiniLMv2`.
3. **المرحلة 2 (المعلّم `Gemma 4 31B bf16` — الفجوات وتقطير المعرفة):**
   - الحزمة تتضمن بالفعل مخرجات المعلّم المحكّمة والمنقّاة (`distilled_real_parents_sft.jsonl`, `distilled_real_parents_dpo.jsonl`, `reranker_teacher_pairs.jsonl`)، ويمكن تفعيل `RUN_EXTRA_COLAB_TEACHER = True` لتحكيم بقية الأسئلة بالكامل على H100.
4. **المرحلة 3 (Fine-Tuning محرك البحث `Reranker` على لهجات الأهالي):** تدريب `mmarco-mMiniLMv2` على صياغات الآباء والأمهات الحقيقية بدون زيادة حجم الموديل.
5. **المرحلة 4 (تدريب `tg-tutor:v6` على `Gemma 4 E4B` بدقة `bf16` SFT + `DPO` وتصدير ورفع `GGUF`):** تدريب النموذج الجديد المخصص للسيرفر المنزلي (`tg-home`)، دمجه مع مرحلة التفضيل المباشر (`DPO`)، تحويله إلى `tg-tutor-v6-Q4_K_M.gguf`، ورفعه تلقائياً إلى المستودع الخاص `khaledhamdy/tg-tutor-v6-gguf`.

In [ ]:
# ── 1. التحقق من كرت الشاشة (H100 / A100 80GB) وتثبيت المكتبات ─────────────
!nvidia-smi

!pip install -qU \
    "transformers>=4.50.0" \
    "trl>=0.16.0" \
    "peft>=0.15.0" \
    "accelerate>=1.4.0" \
    "datasets>=3.3.0" \
    "bitsandbytes>=0.45.3" \
    "sentence-transformers>=3.4.0" \
    "huggingface_hub>=0.29.0" \
    "sentencepiece" \
    "protobuf"

import torch
assert torch.cuda.is_available(), "❌ يرجى تفعيل GPU (A100 أو H100) من Runtime -> Change runtime type"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | bf16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# ── 2. تحميل حزمة المربّي تلقائياً من المستودع الخاص على HuggingFace (أو Drive) ──
# أضف مفتاح HF_TOKEN في Colab Secrets (أيقونة المفتاح 🔑 على اليسار) ليتم سحب البيانات ورفع الموديل تلقائياً بدون أي رفع يدوي!
import os, zipfile, json
from pathlib import Path
from huggingface_hub import hf_hub_download, login

HF_DATASET_REPO = "khaledhamdy/tg-ultra-v6-private"
HF_MODEL_OUTPUT_REPO = "khaledhamdy/tg-tutor-v6-gguf"

hf_token = os.environ.get("HF_TOKEN", "").strip()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = (userdata.get("HF_TOKEN") or "").strip()
    except Exception:
        pass
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
    print("🔑 تم تسجيل الدخول في HuggingFace بنجاح!")

WORK_DIR = Path("/content/tg_ultra")
WORK_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_ZIP = Path("/content/tg_colab_ultra_bundle.zip")

if not BUNDLE_ZIP.exists() and hf_token:
    try:
        print(f"📥 جاري سحب الحزمة تلقائياً من {HF_DATASET_REPO}...")
        downloaded = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            filename="tg_colab_ultra_bundle.zip",
            local_dir="/content",
            token=hf_token,
        )
        BUNDLE_ZIP = Path(downloaded)
    except Exception as e:
        print(f"⚠️ تعذر التحميل من HuggingFace ({e})، جاري البحث في Google Drive أو الرفع اليدوي...")

if not BUNDLE_ZIP.exists():
    drive_candidate = Path("/content/drive/MyDrive/tg_colab_ultra_bundle.zip")
    if drive_candidate.exists():
        BUNDLE_ZIP = drive_candidate
    else:
        from google.colab import files
        print("📤 ارفع ملف ops/data/kaggle_upload_ultra/tg_colab_ultra_bundle.zip:")
        uploaded = files.upload()
        BUNDLE_ZIP = Path(list(uploaded.keys())[0])

with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
    zf.extractall(WORK_DIR)

for p in sorted(WORK_DIR.iterdir()):
    print(f"  📄 {p.name:38s} ({p.stat().st_size / (1024*1024):.2f} MB)")

In [ ]:
# ── 3. المرحلة 1: فحص الاسترجاع والـ Reranker على الـ GPU لكل أسئلة الأهالي ──
# يعمل على A100/H100 في أقل من 10 ثوانٍ بدلاً من 40 دقيقة على الـ CPU
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

kb_units = [json.loads(line) for line in (WORK_DIR / "kb_units_full.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
parent_turns = [json.loads(line) for line in (WORK_DIR / "real_parent_questions.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]

print(f"📚 الوحدات المعرفية: {len(kb_units)} | 💬 رسائل الآباء والأمهات: {len(parent_turns)}")

embedder = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1", max_length=256, device="cuda")

unit_passages = [f"passage: {u['behavior_type']} — {u['text_simplified']}" for u in kb_units]
unit_embs = embedder.encode(unit_passages, batch_size=128, normalize_embeddings=True, show_progress_bar=True)

# نختار الأسئلة العفوية وأسئلة نصيحة اليوم الفريدة
organic_turns = [t for t in parent_turns if t["category"] in ("organic_parent_question", "daily_tip_card")]
unique_by_hash = {}
for t in organic_turns:
    unique_by_hash.setdefault(t["question_hash"], t)
unique_questions = list(unique_by_hash.values())
print(f"🔍 فحص {len(unique_questions)} سؤالاً فريداً من الآباء والأمهات على الـ GPU...")

q_embs = embedder.encode(
    [f"query: {t['question']}" for t in unique_questions],
    batch_size=128, normalize_embeddings=True, show_progress_bar=True
)
sim_matrix = np.matmul(q_embs, unit_embs.T)

# نأخذ أعلى 4 وحدات لكل سؤال ونحسب درجة الـ Cross-Encoder في دفعة واحدة على الـ GPU
cross_pairs = []
cross_meta = []
for q_idx, t in enumerate(unique_questions):
    top_indices = np.argsort(-sim_matrix[q_idx])[:4]
    for u_idx in top_indices:
        u = kb_units[int(u_idx)]
        cross_pairs.append((t["question"], u["text_simplified"][:1200]))
        cross_meta.append((q_idx, int(u_idx), float(sim_matrix[q_idx, u_idx])))

print(f"⚡ حساب درجات الـ Cross-Encoder لـ {len(cross_pairs)} زوج على الـ GPU...")
ce_scores = reranker.predict(cross_pairs, batch_size=256, show_progress_bar=True)

retrieved_by_q = {i: [] for i in range(len(unique_questions))}
for (q_idx, u_idx, vec_sim), ce_score in zip(cross_meta, ce_scores):
    u = kb_units[u_idx]
    retrieved_by_q[q_idx].append({
        "unit_id": u["id"],
        "title": u["behavior_type"],
        "domain": u["domain"],
        "age_group": u["age_group"],
        "reference": u["reference_info"],
        "vector_sim": round(vec_sim, 4),
        "rerank_score": round(float(ce_score), 4),
        "text": u["text_simplified"][:700],
    })

for q_idx, t in enumerate(unique_questions):
    cands = sorted(retrieved_by_q[q_idx], key=lambda x: -x["rerank_score"])
    t["retrieved_units"] = cands
    t["top_rerank_score"] = cands[0]["rerank_score"] if cands else -99.0

probed_out = WORK_DIR / "real_parent_questions_gpu_probed.jsonl"
with probed_out.open("w", encoding="utf-8") as fh:
    for t in unique_questions:
        fh.write(json.dumps(t, ensure_ascii=False) + "\n")

gaps = [t for t in unique_questions if t["unanswered"] or t["top_rerank_score"] < -3.0]
print(f"✅ اكتمل الفحص! إجمالي الأسئلة المرشحة للتحكيم وتوليد الإجابات الذهبية: {len(gaps)} سؤالاً.")
del embedder, reranker
torch.cuda.empty_cache()

In [ ]:
# ── 4. المرحلة 2: التحقق من بيانات المعلّم Gemma 4 31B (أو توسيعها على H100) ──
# الحزمة تحتوي مسبقاً على مخرجات Gemma 4 31B المحكّمة والمنقّاة.
# اجعل RUN_EXTRA_COLAB_TEACHER = True إذا أردت تشغيل google/gemma-4-31b-it محلياً على H100 لإضافة المزيد.
RUN_EXTRA_COLAB_TEACHER = False

sft_path = WORK_DIR / "distilled_real_parents_sft.jsonl"
dpo_path = WORK_DIR / "distilled_real_parents_dpo.jsonl"
rerank_path = WORK_DIR / "reranker_teacher_pairs.jsonl"

if sft_path.exists() and not RUN_EXTRA_COLAB_TEACHER:
    n_sft = sum(1 for _ in sft_path.open(encoding="utf-8"))
    n_dpo = sum(1 for _ in dpo_path.open(encoding="utf-8")) if dpo_path.exists() else 0
    n_rr = sum(1 for _ in rerank_path.open(encoding="utf-8")) if rerank_path.exists() else 0
    print(f"✅ مخرجات المعلّم Gemma 4 31B جاهزة ومحمّلة من الحزمة:")
    print(f"   • SFT Gold Answers: {n_sft}")
    print(f"   • DPO Preference Pairs: {n_dpo}")
    print(f"   • Reranker Teacher Pairs: {n_rr}")
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    TEACHER_MODEL = "google/gemma-4-31b-it"
    MAX_TEACHER_QUESTIONS = 400
    print(f"🧠 جاري تحميل المعلّم {TEACHER_MODEL} بدقة bfloat16...")
    teacher_tok = AutoTokenizer.from_pretrained(TEACHER_MODEL)
    teacher_model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
    teacher_model.eval()
    del teacher_model, teacher_tok
    torch.cuda.empty_cache()

In [ ]:
# ── 5. المرحلة 3: تدريب الـ Cross-Encoder Reranker على لهجات الأهالي ────────
# يحسّن دقة التقاط الوحدة المعرفية لأسئلة العامية والفصحى دون زيادة حجم النموذج
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, CrossEncoder
import shutil

rerank_samples = [
    InputExample(texts=[r["question"], r["passage"]], label=float(r["label"]))
    for r in [json.loads(l) for l in (WORK_DIR / "reranker_teacher_pairs.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
]

if len(rerank_samples) >= 32:
    print(f"🎯 جاري تدريب الـ Reranker على {len(rerank_samples)} زوج من أسئلة الأهالي الحقيقية...")
    ce_model = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1", num_labels=1, max_length=256, device="cuda")
    train_dataloader = DataLoader(rerank_samples, shuffle=True, batch_size=32)
    reranker_out_dir = str(WORK_DIR / "tg-reranker-v2")
    ce_model.fit(
        train_dataloader=train_dataloader,
        epochs=3,
        warmup_steps=max(10, int(len(train_dataloader) * 0.1)),
        output_path=reranker_out_dir,
    )
    ce_model.save(reranker_out_dir)
    shutil.make_archive("/content/tg-reranker-v2", "zip", reranker_out_dir)
    print("✅ تم حفظ الـ Reranker المطور في /content/tg-reranker-v2.zip")
    del ce_model
    torch.cuda.empty_cache()
else:
    print("ℹ️ عدد الأزواج أقل من 32 — تخطي تدريب الـ Reranker.")

In [ ]:
# ── 6. المرحلة 4: تدريب نموذج المربّي الجديد (tg-tutor:v6) على Gemma 4 E4B (SFT + DPO بـ bf16) ──
# على كرت H100 / A100 80GB ندرّب بدقة bfloat16 الأصلية بدون فقدان الدقة في 4-bit، ثم نطبّق DPO!
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig

STUDENT_MODEL = "google/gemma-4-e4b-it"  # سريع جداً على سيرفر tg-home (CPU)
USE_FULL_FINETUNE = False  # اجعلها True لعمل Full Fine-Tune كامل الأوزان على 80GB، أو False لـ High-Rank bf16 LoRA (r=64)
OUT_SFT_DIR = WORK_DIR / "tg-tutor-v6-sft-lora"
OUT_ADAPTER_DIR = WORK_DIR / "tg-tutor-v6-lora"
MERGED_MODEL_DIR = WORK_DIR / "tg-tutor-v6-merged"

tok = AutoTokenizer.from_pretrained(STUDENT_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def render_chat(example):
    return {"text": tok.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

# tg_train_ultra_v6.jsonl مدمج فيه مسبقاً الإجابات الذهبية بوزن 3x
ds_train = load_dataset("json", data_files=str(WORK_DIR / "tg_train_ultra_v6.jsonl"), split="train")
ds_val = load_dataset("json", data_files=str(WORK_DIR / "tg_val_ultra_v6.jsonl"), split="train")

ds_train = ds_train.map(render_chat, remove_columns=ds_train.column_names)
ds_val = ds_val.map(render_chat, remove_columns=ds_val.column_names)
print(f"📊 حجم التدريب النهائي (SFT): {len(ds_train)} | التحقق: {len(ds_val)}")

student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
student_model.config.use_cache = False

peft_config = None if USE_FULL_FINETUNE else LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_args = SFTConfig(
    output_dir=str(OUT_SFT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=16,   # 80GB VRAM تسمح بـ batch_size=16 بسهولة!
    gradient_accumulation_steps=2,    # Effective batch = 32
    learning_rate=2e-5 if USE_FULL_FINETUNE else 1.5e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=2,
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=True,                     # يضاعف سرعة التدريب 3x على A100/H100
    report_to="none",
)

trainer = SFTTrainer(
    model=student_model,
    args=sft_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    peft_config=peft_config,
    processing_class=tok,
)
trainer.train()
trainer.save_model(str(OUT_SFT_DIR))
tok.save_pretrained(str(OUT_SFT_DIR))
print("✅ اكتمل تدريب SFT بنجاح!")

# ── المرحلة 4-ب: التفضيل المباشر (DPO) على أزواج أسئلة الأهالي الحقيقية ──────
dpo_file = WORK_DIR / "distilled_real_parents_dpo.jsonl"
if dpo_file.exists() and dpo_file.stat().st_size > 0:
    ds_dpo = load_dataset("json", data_files=str(dpo_file), split="train")
    print(f"🎯 جاري تشغيل مرحلة DPO على {len(ds_dpo)} زوج تفضيل (إجابة Gemma 4 31B الذهبية مقابل الإجابة القديمة)...")
    dpo_args = DPOConfig(
        output_dir=str(OUT_ADAPTER_DIR),
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        beta=0.1,
        bf16=True,
        logging_steps=10,
        save_strategy="no",
        max_length=1024,
        max_prompt_length=384,
        report_to="none",
    )
    dpo_trainer = DPOTrainer(
        model=trainer.model,
        ref_model=None,
        args=dpo_args,
        train_dataset=ds_dpo,
        processing_class=tok,
    )
    dpo_trainer.train()
    dpo_trainer.save_model(str(OUT_ADAPTER_DIR))
    tok.save_pretrained(str(OUT_ADAPTER_DIR))
    print("✅ اكتمل تدريب DPO بنجاح!")
else:
    OUT_ADAPTER_DIR = OUT_SFT_DIR

In [ ]:
# ── 7. دمج الأوزان، التحويل إلى GGUF (Q4_K_M)، والرفع التلقائي إلى HuggingFace ──
from peft import AutoPeftModelForCausalLM
from huggingface_hub import HfApi

if USE_FULL_FINETUNE:
    MERGED_MODEL_DIR = OUT_ADAPTER_DIR
else:
    print("🔄 جاري دمج LoRA مع النموذج الأساسي بدقة float16...")
    merged = AutoPeftModelForCausalLM.from_pretrained(
        str(OUT_ADAPTER_DIR),
        torch_dtype=torch.float16,
        device_map="auto",
    )
    merged = merged.merge_and_unload()
    merged.save_pretrained(str(MERGED_MODEL_DIR), safe_serialization=True)
    tok.save_pretrained(str(MERGED_MODEL_DIR))
    del merged
    torch.cuda.empty_cache()

!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -B /content/llama.cpp/build /content/llama.cpp && cmake --build /content/llama.cpp/build --config Release -j 4 --target llama-quantize
!pip install -q -r /content/llama.cpp/requirements.txt

GGUF_F16 = "/content/tg-tutor-v6-f16.gguf"
GGUF_Q4 = "/content/tg-tutor-v6-Q4_K_M.gguf"

!python3 /content/llama.cpp/convert_hf_to_gguf.py {MERGED_MODEL_DIR} --outfile {GGUF_F16} --outtype f16
!/content/llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_Q4} Q4_K_M

modelfile_text = """FROM ./tg-tutor-v6-Q4_K_M.gguf

SYSTEM "أنت «المربّي»، مساعد تربوي عربي للأهل متخصص في التربية الإسلامية ونمو الطفل وصحته وأمانه الرقمي. أجب بالعربية الفصحى الميسرة بوضوح وتعاطف، وقدّم 3 إلى 4 خطوات عملية مرقمة قابلة للتنفيذ اليوم مستندة إلى مصادر موثوقة، واختم بسطر 📚 المصادر. لا تعطِ تشخيصاً طبياً ملزماً ولا جرعات أدوية ولا فتوى شخصية قاطعة، ولا تحل إلى متخصص إلا عند وجود خطر طبي أو نفسي حقيقي."

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
"""
MODELFILE_PATH = Path("/content/Modelfile.tg-tutor-v6")
MODELFILE_PATH.write_text(modelfile_text, encoding="utf-8")

if hf_token:
    print(f"☁️ جاري رفع النموذج المكمّم والـ Reranker تلقائياً إلى المستودع الخاص {HF_MODEL_OUTPUT_REPO}...")
    api = HfApi(token=hf_token)
    api.create_repo(repo_id=HF_MODEL_OUTPUT_REPO, repo_type="model", private=True, exist_ok=True)
    for local_f in (GGUF_Q4, str(MODELFILE_PATH), "/content/tg-reranker-v2.zip"):
        if Path(local_f).exists():
            api.upload_file(
                path_or_fileobj=local_f,
                path_in_repo=Path(local_f).name,
                repo_id=HF_MODEL_OUTPUT_REPO,
                repo_type="model",
            )
            print(f"  ✅ تم رفع {Path(local_f).name} -> {HF_MODEL_OUTPUT_REPO}")

print(f"🎉 جاهز للنشر على سيرفر البيت (tg-home): شغّل الأمر التالي على tg-home:")
print("  HF_TOKEN=<your_token> ~/deploy_from_hf.sh")